# Week 1 — Data Collection & Cleaning
## *Do LLMs Read 10-Ks Better Than Dictionaries?*

**数据来源：**
1. **SEC EDGAR** — 10-K全文 & filing dates（免费API）
2. **WRDS / Compustat** — CIK-ticker对照、公司基本面（at, sic）
3. **WRDS / CRSP** — 日频股票收益（ret, vwretd）

**目标范围：** 2010–2020，S&P 1500成分股，约 50,000–80,000 份 10-K

---
### 流程图
```
EDGAR Submissions API
  → 10-K filing list (CIK, accession, date_filed)
      ↓
EDGAR Full-Text
  → 下载 MD&A 段落全文
      ↓
WRDS Compustat
  → 公司基本面 + CIK-ticker/gvkey 对照
      ↓
WRDS CRSP
  → 日频收益 (用于后续 Week 3 CAR 计算)
      ↓
合并 → master_panel.parquet
```

## 0. 环境配置 & 依赖安装

In [1]:
# 安装依赖（首次运行）
# !pip install wrds pandas pyarrow tqdm requests beautifulsoup4 lxml ratelimit

import os
import re
import time
import json
import requests
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
from datetime import datetime, date

warnings.filterwarnings('ignore')

# ----------- 全局配置 -----------
START_YEAR   = 2010
END_YEAR     = 2020
DATA_DIR     = Path("data")
RAW_DIR      = DATA_DIR / "raw"
FILINGS_DIR  = DATA_DIR / "filings"   # 存放 MD&A 文本
OUTPUT_DIR   = DATA_DIR / "processed"

for d in [RAW_DIR, FILINGS_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# EDGAR API headers（SEC要求标注 User-Agent，否则会被限流）
HEADERS = {
    "User-Agent": "OliviaWang oliviawang1231@outlook.com",  # ← 改成你自己的
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

print("✅ 环境配置完成")
print(f"数据目录: {DATA_DIR.resolve()}")

✅ 环境配置完成
数据目录: /Users/yulinwang/Desktop/macs final project/data


/opt/anaconda3/envs/111/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Part 1: SEC EDGAR — 获取 10-K Filing 列表

**策略：**
- 方法 A（推荐）：用 EDGAR `submissions` API，逐公司拉 filing history
- 方法 B（备用）：用 EDGAR full-text search API 批量搜索

先用方法 B 快速获取所有 10-K 的 accession number 列表（高效），
再用方法 A 补充 CIK→ticker 对照和详细元数据。

### 1.1 方法 A：用 EDGAR Submissions API 逐 CIK 拉 filing 记录

In [ ]:
# ── Helper：调用 EDGAR Submissions API ──────────────────────────────────────

BASE_SUBMISSIONS = "https://data.sec.gov/submissions/CIK{cik}.json"
BASE_COMPANY     = "https://www.sec.gov/cgi-bin/browse-edgar"
RATE_SLEEP       = 0.12   # SEC限制：10 req/s，保守用 8 req/s

def get_submissions(cik: str) -> dict:
    """根据 10位 CIK 获取公司所有 filing 记录"""
    cik_padded = str(cik).zfill(10)
    url = BASE_SUBMISSIONS.format(cik=cik_padded)
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    time.sleep(RATE_SLEEP)
    return r.json()

def extract_10k_filings(submissions: dict, start_year: int, end_year: int) -> pd.DataFrame:
    """
    从 submissions JSON 中提取 10-K / 10-K405 filing 记录，
    返回 DataFrame: [cik, company_name, form_type, date_filed, accession_no]
    """
    filings = submissions.get("filings", {}).get("recent", {})
    if not filings:
        return pd.DataFrame()

    df = pd.DataFrame(filings)
    # 只保留 10-K 类型
    mask = df["form"].isin(["10-K", "10-K405", "10-KSB"])
    df = df[mask].copy()
    df["date_filed"] = pd.to_datetime(df["filingDate"])
    # 年份过滤
    df = df[(df["date_filed"].dt.year >= start_year) &
            (df["date_filed"].dt.year <= end_year)]

    df["cik"]          = submissions["cik"]
    df["company_name"] = submissions.get("name", "")
    df["ticker"]       = submissions.get("tickers", [None])[0]  # 取第一个 ticker
    df["sic"]          = submissions.get("sic", None)
    df["sic_desc"]     = submissions.get("sicDescription", "")
    df["state"]        = submissions.get("stateOfIncorporation", "")

    # 重命名对齐
    df = df.rename(columns={
        "form": "form_type",
        "accessionNumber": "accession_no",
        "primaryDocument": "primary_doc",
        "reportDate": "period_of_report"
    })

    keep_cols = ["cik", "ticker", "company_name", "sic", "sic_desc", "state",
                 "form_type", "date_filed", "period_of_report",
                 "accession_no", "primary_doc"]
    return df[keep_cols].reset_index(drop=True)

print("✅ EDGAR helper 函数定义完成")

In [ ]:
# ── 获取 EDGAR 全部上市公司 CIK 列表 ──────────────────────────────────────────
# SEC 提供一个公司列表 JSON，包含所有注册公司的 CIK + ticker

COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"

headers_main = {"User-Agent": "YourName YourEmail@uchicago.edu"}  # ← 改

r = requests.get(COMPANY_TICKERS_URL, headers=headers_main, timeout=30)
company_tickers = pd.DataFrame(r.json()).T
company_tickers.columns = ["cik", "ticker", "company_name"]
company_tickers["cik"] = company_tickers["cik"].astype(str).str.zfill(10)

print(f"EDGAR 注册公司总数: {len(company_tickers):,}")
company_tickers.head()

### 1.2 从 WRDS/Compustat 获取 S&P 1500 成分股列表

> **注意**：下面的 WRDS 代码需要你在有 WRDS 账号的环境下运行。  
> 如果是本地运行，先 `pip install wrds`，然后用你的 WRDS 账号登录。

In [ ]:
# ── 连接 WRDS ─────────────────────────────────────────────────────────────────
import wrds

# 首次运行会提示输入账号密码，之后会缓存 pgpass
db = wrds.Connection(wrds_username="your_wrds_username")  # ← 改成你的 WRDS 账号
print("✅ WRDS 连接成功")

In [ ]:
# ── 从 Compustat 获取 S&P 1500 成分股 ────────────────────────────────────────
# spidx = S&P composite index constituents table
# indextype: 'SPI' = S&P 500, 'MID' = S&P 400 MidCap, 'SML' = S&P 600 SmallCap

sp1500_query = """
    SELECT DISTINCT
        a.gvkey,
        a.tic        AS ticker,
        a.conm       AS company_name,
        a.cik,
        a.sic,
        a.naics,
        a.fic        AS country,
        a.exchg      AS exchange
    FROM comp.company AS a
    INNER JOIN comp.idxcst_his AS b
        ON a.gvkey = b.gvkey
    WHERE
        b.gvkeyx IN ('000003','000009','000010')  -- SP500, SP400, SP600
        AND b.from <= '2020-12-31'
        AND (b.thru >= '2010-01-01' OR b.thru IS NULL)
        AND a.fic = 'USA'        -- 只要美国公司
        AND a.cik IS NOT NULL    -- 必须有 CIK 才能匹配 EDGAR
"""

sp1500 = db.raw_sql(sp1500_query)
sp1500["cik"] = sp1500["cik"].astype(str).str.strip().str.zfill(10)
sp1500 = sp1500.drop_duplicates(subset=["gvkey"])

print(f"S&P 1500 成分股（去重后）: {len(sp1500):,} 家公司")
print(f"有 CIK 记录: {sp1500['cik'].notna().sum():,}")
sp1500.head()

In [ ]:
# 保存 S&P 1500 名单
sp1500.to_csv(RAW_DIR / "sp1500_universe.csv", index=False)
print(f"✅ 已保存: {RAW_DIR / 'sp1500_universe.csv'}")

# 提取目标 CIK 集合
target_ciks = set(sp1500["cik"].dropna().unique())
print(f"目标 CIK 数量: {len(target_ciks):,}")

### 1.3 批量从 EDGAR Submissions API 拉取 10-K Filing 列表

In [ ]:
# ── 批量拉取（带断点续跑）──────────────────────────────────────────────────────
CHECKPOINT_FILE = RAW_DIR / "edgar_filings_checkpoint.parquet"
ERROR_LOG       = RAW_DIR / "edgar_errors.txt"

# 如果有断点，直接加载
if CHECKPOINT_FILE.exists():
    filings_df = pd.read_parquet(CHECKPOINT_FILE)
    done_ciks  = set(filings_df["cik"].unique())
    print(f"断点续跑：已完成 {len(done_ciks):,} 家，剩余 {len(target_ciks - done_ciks):,} 家")
else:
    filings_df = pd.DataFrame()
    done_ciks  = set()

remaining_ciks = sorted(target_ciks - done_ciks)
all_new_filings = []
errors = []

for i, cik in enumerate(tqdm(remaining_ciks, desc="拉取 EDGAR submissions")):
    try:
        subs = get_submissions(cik)
        df_cik = extract_10k_filings(subs, START_YEAR, END_YEAR)
        if len(df_cik) > 0:
            all_new_filings.append(df_cik)
    except Exception as e:
        errors.append(f"{cik}: {str(e)}")
        continue

    # 每 500 家存一次断点
    if (i + 1) % 500 == 0 and all_new_filings:
        new_df = pd.concat(all_new_filings, ignore_index=True)
        filings_df = pd.concat([filings_df, new_df], ignore_index=True)
        filings_df.to_parquet(CHECKPOINT_FILE)
        all_new_filings = []
        print(f"  断点保存：当前共 {len(filings_df):,} 条记录")

# 最后一批
if all_new_filings:
    new_df = pd.concat(all_new_filings, ignore_index=True)
    filings_df = pd.concat([filings_df, new_df], ignore_index=True)
    filings_df.to_parquet(CHECKPOINT_FILE)

# 保存错误日志
with open(ERROR_LOG, "w") as f:
    f.write("\n".join(errors))

print(f"\n✅ EDGAR filings 拉取完成")
print(f"   总 filing 记录: {len(filings_df):,}")
print(f"   涉及公司数:      {filings_df['cik'].nunique():,}")
print(f"   错误数:          {len(errors):,}")

In [ ]:
# 查看 filing 分布
filings_df["year"] = pd.to_datetime(filings_df["date_filed"]).dt.year
print("各年 10-K 数量：")
print(filings_df.groupby("year").size().to_string())

print("\n各 form type 分布：")
print(filings_df["form_type"].value_counts())

---
## Part 2: 下载 10-K MD&A 段落全文

**策略：**
- 用 accession number 构造 EDGAR 文件 URL
- 下载 `.htm` / `.txt` 主文件
- 用正则 + BeautifulSoup 提取 **Item 7 (MD&A)** 段落
- 存为 `.txt` 文件（按 CIK/year 命名）

In [ ]:
# ── Helper：构造 EDGAR 文件 URL ──────────────────────────────────────────────

BASE_ARCHIVE = "https://www.sec.gov/Archives/edgar/data"

def build_filing_url(cik: str, accession_no: str, primary_doc: str) -> str:
    """构造 10-K 主文件的下载 URL"""
    acc_no_clean = accession_no.replace("-", "")
    return f"{BASE_ARCHIVE}/{int(cik)}/{acc_no_clean}/{primary_doc}"

def build_index_url(cik: str, accession_no: str) -> str:
    """构造 filing index 页面 URL（可以看到文件列表）"""
    acc_no_clean = accession_no.replace("-", "")
    return f"{BASE_ARCHIVE}/{int(cik)}/{acc_no_clean}/{accession_no}-index.htm"


def fetch_filing_text(url: str, retries: int = 3) -> str | None:
    """下载 filing 文件，返回原始文本"""
    headers = {"User-Agent": "YourName YourEmail@uchicago.edu"}  # ← 改
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                time.sleep(RATE_SLEEP)
                return r.text
            elif r.status_code == 404:
                return None  # 文件不存在
        except requests.RequestException:
            time.sleep(2 ** attempt)   # 指数退避
    return None


print("✅ 文件下载 helper 定义完成")

In [ ]:
# ── Helper：从 10-K HTML/TXT 中提取 MD&A 段落 ─────────────────────────────────

# Item 7 的各种写法
ITEM7_PATTERNS = [
    r"item\s+7[^a-z]*management['\'s]*\s+discussion",
    r"item\s+7\.\s+management",
    r"management['\'s]+\s+discussion\s+and\s+analysis",
]
# Item 7A 标志着 MD&A 结束
ITEM7A_PATTERNS = [
    r"item\s+7a",
    r"quantitative.*qualitative.*disclosures.*market\s+risk",
]

def extract_mda(raw_text: str, max_chars: int = 150_000) -> str:
    """
    从 10-K 原始文本中提取 MD&A（Item 7）段落。
    返回纯文本，截断在 max_chars 字符（约 30,000 词）。
    """
    # 1. 去除 HTML 标签
    if "<html" in raw_text.lower() or "<htm" in raw_text.lower():
        soup = BeautifulSoup(raw_text, "lxml")
        # 移除 script/style
        for tag in soup(["script", "style", "table"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
    else:
        text = raw_text

    # 2. 统一空白
    text = re.sub(r"\s+", " ", text)
    text_lower = text.lower()

    # 3. 找 Item 7 起始位置
    start_pos = None
    for pat in ITEM7_PATTERNS:
        m = re.search(pat, text_lower)
        if m:
            # 跳过目录里的（通常在前 20% 文本，且很短）
            if m.start() > len(text) * 0.05:
                start_pos = m.start()
                break

    if start_pos is None:
        return ""  # 没找到 MD&A

    # 4. 找 Item 7A（MD&A 结束）
    end_pos = None
    for pat in ITEM7A_PATTERNS:
        m = re.search(pat, text_lower[start_pos + 100:])
        if m:
            end_pos = start_pos + 100 + m.start()
            break

    if end_pos is None or (end_pos - start_pos) < 500:
        # 没找到结束标志，取 Item 7 后的 max_chars 字符
        end_pos = start_pos + max_chars

    mda = text[start_pos:end_pos].strip()
    return mda[:max_chars]   # 截断


print("✅ MD&A 提取函数定义完成")

# 快速测试
test_text = "ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS The company performed well. ITEM 7A. QUANTITATIVE DISCLOSURES"
print("测试提取:", extract_mda(test_text)[:80])

In [ ]:
# ── 批量下载 MD&A（带断点续跑）──────────────────────────────────────────────────

MDA_META_FILE = RAW_DIR / "mda_metadata.parquet"

if MDA_META_FILE.exists():
    mda_meta = pd.read_parquet(MDA_META_FILE)
    done_accessions = set(mda_meta["accession_no"].unique())
    print(f"断点续跑：已下载 {len(done_accessions):,} 份")
else:
    mda_meta = pd.DataFrame()
    done_accessions = set()

# 只处理还没下载的
filings_todo = filings_df[~filings_df["accession_no"].isin(done_accessions)].copy()
print(f"待下载: {len(filings_todo):,} 份 10-K")

new_meta_rows = []
batch_size = 1000  # 每 1000 份存一次断点

for i, row in enumerate(tqdm(filings_todo.itertuples(), total=len(filings_todo),
                              desc="下载 MD&A")):
    cik      = str(row.cik).zfill(10)
    acc_no   = row.accession_no
    pri_doc  = str(row.primary_doc) if pd.notna(row.primary_doc) else ""

    # 输出目录：data/filings/{cik}/{year}/
    year     = pd.to_datetime(row.date_filed).year
    save_dir = FILINGS_DIR / cik / str(year)
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"{acc_no.replace('-','')}.txt"

    meta_row = {
        "cik": cik, "accession_no": acc_no, "date_filed": row.date_filed,
        "year": year, "mda_path": str(save_path),
        "mda_len": 0, "status": "skipped"
    }

    # 如果文件已存在，跳过下载
    if save_path.exists():
        meta_row["mda_len"] = save_path.stat().st_size
        meta_row["status"]  = "cached"
        new_meta_rows.append(meta_row)
        continue

    # 构造 URL
    if pri_doc:
        url = build_filing_url(cik, acc_no, pri_doc)
    else:
        # 没有 primary_doc，先查 index 页面找 htm 文件
        idx_url  = build_index_url(cik, acc_no)
        idx_text = fetch_filing_text(idx_url)
        if idx_text is None:
            meta_row["status"] = "error_index"
            new_meta_rows.append(meta_row)
            continue
        # 从 index 页面找 10-K 主文件
        htm_links = re.findall(r'href="([^"]+\.htm)"', idx_text, re.IGNORECASE)
        if not htm_links:
            meta_row["status"] = "error_no_htm"
            new_meta_rows.append(meta_row)
            continue
        url = f"https://www.sec.gov{htm_links[0]}" if htm_links[0].startswith("/") else htm_links[0]

    # 下载文件
    raw_text = fetch_filing_text(url)
    if raw_text is None:
        meta_row["status"] = "error_download"
        new_meta_rows.append(meta_row)
        continue

    # 提取 MD&A
    mda_text = extract_mda(raw_text)
    if len(mda_text) < 200:   # MD&A 太短，可能是提取失败
        meta_row["status"] = "error_mda_short"
        new_meta_rows.append(meta_row)
        continue

    # 保存
    save_path.write_text(mda_text, encoding="utf-8")
    meta_row["mda_len"] = len(mda_text)
    meta_row["status"]  = "ok"
    new_meta_rows.append(meta_row)

    # 定期存断点
    if (i + 1) % batch_size == 0:
        new_meta_df = pd.DataFrame(new_meta_rows)
        mda_meta = pd.concat([mda_meta, new_meta_df], ignore_index=True)
        mda_meta.to_parquet(MDA_META_FILE)
        new_meta_rows = []
        ok_cnt = (mda_meta["status"] == "ok").sum()
        print(f"  进度 {i+1:,}: 成功 {ok_cnt:,} 份")

# 最后一批
if new_meta_rows:
    mda_meta = pd.concat([mda_meta, pd.DataFrame(new_meta_rows)], ignore_index=True)
    mda_meta.to_parquet(MDA_META_FILE)

print("\n✅ MD&A 批量下载完成")
print(mda_meta["status"].value_counts())

---
## Part 3: WRDS Compustat — 公司基本面数据

In [ ]:
# ── 从 Compustat Annual (funda) 拉取基本面 ────────────────────────────────────
# 变量说明：
#   at   = total assets（用于计算公司规模 log(at)）
#   ceq  = common equity
#   ni   = net income
#   sale = revenue
#   bm   = book-to-market（需要自算：ceq / market cap）
#   sic  = SIC 行业代码

# 先获取目标公司的 gvkey 列表
gvkey_list = "', '".join(sp1500["gvkey"].astype(str).tolist())

compustat_query = f"""
    SELECT
        gvkey,
        datadate,
        fyear,
        tic         AS ticker,
        conm        AS company_name,
        sic,
        naics,
        at          AS total_assets,
        ceq         AS book_equity,
        ni          AS net_income,
        sale        AS revenue,
        dltt        AS long_term_debt,
        csho        AS shares_out,
        prcc_f      AS price_fiscal_year_end,
        cik
    FROM comp.funda
    WHERE
        gvkey IN ('{gvkey_list}')
        AND fyear BETWEEN {START_YEAR} AND {END_YEAR}
        AND indfmt  = 'INDL'      -- 工业格式（排除金融行业特殊格式）
        AND datafmt = 'STD'       -- 标准格式
        AND popsrc  = 'D'         -- 主要来源
        AND consol  = 'C'         -- 合并报表
    ORDER BY gvkey, fyear
"""

compustat = db.raw_sql(compustat_query)
compustat["cik"] = compustat["cik"].astype(str).str.strip().str.zfill(10)

print(f"✅ Compustat 数据: {len(compustat):,} 条公司-年度观测")
print(f"   涉及公司: {compustat['gvkey'].nunique():,}")
print(f"   年份范围: {compustat['fyear'].min()} – {compustat['fyear'].max()}")
compustat.head()

In [ ]:
# ── 计算控制变量 ──────────────────────────────────────────────────────────────

compustat["log_assets"] = np.log(compustat["total_assets"].clip(lower=1e-6))
compustat["market_cap"] = compustat["shares_out"] * compustat["price_fiscal_year_end"]
compustat["log_mktcap"] = np.log(compustat["market_cap"].clip(lower=1e-6))
compustat["bm_ratio"]   = compustat["book_equity"] / compustat["market_cap"].clip(lower=1e-6)
compustat["roa"]        = compustat["net_income"] / compustat["total_assets"].clip(lower=1e-6)
compustat["leverage"]   = compustat["long_term_debt"] / compustat["total_assets"].clip(lower=1e-6)

# SIC 一级行业分类（Fama-French 12 industries 简版）
def sic_to_ff12(sic):
    sic = int(sic) if pd.notna(sic) else 0
    if 100  <= sic <= 999:  return "Agriculture"
    if 1000 <= sic <= 1499: return "Mining"
    if 1500 <= sic <= 1799: return "Construction"
    if 2000 <= sic <= 3999: return "Manufacturing"
    if 4000 <= sic <= 4799: return "Transportation"
    if 4800 <= sic <= 4999: return "Utilities"
    if 5000 <= sic <= 5199: return "Wholesale"
    if 5200 <= sic <= 5999: return "Retail"
    if 6000 <= sic <= 6799: return "Finance"
    if 7000 <= sic <= 8999: return "Services"
    return "Other"

compustat["ff_industry"] = compustat["sic"].apply(sic_to_ff12)

print("行业分布：")
print(compustat.groupby("ff_industry").size().sort_values(ascending=False))

# 保存
compustat.to_parquet(RAW_DIR / "compustat_fundamentals.parquet")
print("\n✅ Compustat 数据已保存")

---
## Part 4: WRDS CRSP — 日频股票收益数据

> **注意**：CRSP 日频数据量很大。建议分批查询，或只拉 event window 附近的数据。  
> 这里我们拉所有目标股票的全量日频数据（2009–2021，多一年用于 estimation window）。

In [ ]:
# ── Step 1: 获取 CRSP permno ← Compustat gvkey 对照表 ─────────────────────────
# WRDS 提供 ccmxpf_linktable 做 Compustat-CRSP 连接

link_query = f"""
    SELECT DISTINCT
        a.gvkey,
        b.lpermno   AS permno,
        b.linktype,
        b.linkprim,
        b.linkdt,
        b.linkenddt
    FROM comp.company AS a
    INNER JOIN crsp.ccmxpf_linktable AS b
        ON a.gvkey = b.gvkey
    WHERE
        a.gvkey IN ('{gvkey_list}')
        AND b.linktype IN ('LU', 'LC')    -- 高质量链接
        AND b.linkprim IN ('P', 'C')      -- 主要/唯一链接
        AND b.lpermno IS NOT NULL
"""

ccm_link = db.raw_sql(link_query)
print(f"✅ CCM Link Table: {len(ccm_link):,} 条链接记录")
print(f"   唯一 gvkey: {ccm_link['gvkey'].nunique():,}")
print(f"   唯一 permno: {ccm_link['permno'].nunique():,}")

ccm_link.to_parquet(RAW_DIR / "ccm_link.parquet")
ccm_link.head()

In [ ]:
# ── Step 2: 从 CRSP 日频数据库拉收益 ─────────────────────────────────────────
# 变量说明：
#   ret    = 个股日收益（含股息）
#   vwretd = CRSP 等权市场收益（value-weighted，含股息），用于超额收益计算
#   shrout = 流通股数（用于计算市值）
#   prc    = 收盘价（绝对值，因为 CRSP 负数表示 bid-ask 均值）
#   vol    = 交易量

permno_list = ccm_link["permno"].dropna().astype(int).unique()
permno_str  = ", ".join(str(p) for p in permno_list)

# 拉 2009-2021 日频数据（estimation window 需要事件前 250 个交易日）
crsp_query = f"""
    SELECT
        a.permno,
        a.date,
        a.ret,
        a.retx,       -- 不含股息收益（robustness用）
        a.shrout,
        a.prc,
        a.vol,
        b.vwretd,     -- CRSP value-weighted market return
        b.ewretd      -- equal-weighted market return（备用）
    FROM crsp.dsf AS a
    LEFT JOIN crsp.dsi AS b
        ON a.date = b.date
    WHERE
        a.permno IN ({permno_str})
        AND a.date BETWEEN '2009-01-01' AND '2021-12-31'
    ORDER BY a.permno, a.date
"""

print("正在从 CRSP 拉取日频数据（可能需要几分钟）...")
crsp_daily = db.raw_sql(crsp_query, date_cols=["date"])

print(f"✅ CRSP 日频数据: {len(crsp_daily):,} 条")
print(f"   涉及 permno: {crsp_daily['permno'].nunique():,}")
print(f"   日期范围: {crsp_daily['date'].min()} – {crsp_daily['date'].max()}")

# 处理缺失收益
crsp_daily["ret"] = pd.to_numeric(crsp_daily["ret"], errors="coerce")
crsp_daily["prc"] = crsp_daily["prc"].abs()   # CRSP 负数 = 估计价格

# 保存（parquet 压缩，约 300–500 MB）
crsp_daily.to_parquet(RAW_DIR / "crsp_daily.parquet", compression="snappy")
print(f"✅ CRSP 数据已保存")
crsp_daily.head()

---
## Part 5: 数据合并 — 构建 Master Panel

**最终合并逻辑：**
```
filings_df (cik + date_filed)
   ↕ cik
sp1500 → compustat (gvkey + fyear → controls)
   ↕ gvkey → permno (via ccm_link)
crsp_daily (permno + date → returns for CAR)
```

In [ ]:
# ── 5.1 合并 filings + Compustat ─────────────────────────────────────────────

# 从 filings 中只保留成功下载 MD&A 的记录
mda_ok = mda_meta[mda_meta["status"] == "ok"][["cik", "accession_no", "date_filed",
                                                "year", "mda_path", "mda_len"]]

# 合并 SP1500 的 gvkey
sp1500_slim = sp1500[["gvkey", "cik", "ticker"]].drop_duplicates()

panel = mda_ok.merge(
    sp1500_slim,
    on="cik",
    how="inner"   # 只保留 S&P 1500 范围内
)

print(f"合并后（filings × SP1500）: {len(panel):,} 条")

# 合并 Compustat 基本面（用 fyear = 10-K 所属财年）
# EDGAR filing date 通常是会计年度结束后 60-90 天，fyear = date_filed.year - 1（通常）
# 更精确：用 period_of_report 的年份
panel["date_filed"] = pd.to_datetime(panel["date_filed"])
panel["fyear"]      = panel["year"] - 1   # 保守估计：10-K 在次年提交

compustat_slim = compustat[[
    "gvkey", "fyear", "sic", "ff_industry",
    "log_assets", "log_mktcap", "bm_ratio", "roa", "leverage"
]].drop_duplicates(subset=["gvkey", "fyear"])

panel = panel.merge(
    compustat_slim,
    on=["gvkey", "fyear"],
    how="left"
)

print(f"合并后（+ Compustat）: {len(panel):,} 条")
print(f"  基本面匹配率: {panel['log_assets'].notna().mean():.1%}")

In [ ]:
# ── 5.2 合并 CRSP permno ─────────────────────────────────────────────────────

# 处理 CCM link 的有效期（linkdt/linkenddt）
ccm_clean = ccm_link.copy()
ccm_clean["linkdt"]    = pd.to_datetime(ccm_clean["linkdt"], errors="coerce")
ccm_clean["linkenddt"] = pd.to_datetime(ccm_clean["linkenddt"], errors="coerce")
ccm_clean["linkenddt"] = ccm_clean["linkenddt"].fillna(pd.Timestamp("2099-12-31"))

# 为每个 (gvkey, date_filed) 找匹配的 permno
panel_with_permno = panel.merge(
    ccm_clean[["gvkey", "permno", "linkdt", "linkenddt"]],
    on="gvkey",
    how="left"
)

# 过滤：date_filed 必须在 link 有效期内
date_mask = (
    (panel_with_permno["date_filed"] >= panel_with_permno["linkdt"]) &
    (panel_with_permno["date_filed"] <= panel_with_permno["linkenddt"])
)
panel_with_permno = panel_with_permno[date_mask | panel_with_permno["permno"].isna()]

# 去重（一个 gvkey 可能有多个 permno）
panel_with_permno = panel_with_permno.sort_values(
    ["gvkey", "date_filed", "permno"]
).drop_duplicates(subset=["gvkey", "date_filed"], keep="first")

panel = panel_with_permno.drop(columns=["linkdt", "linkenddt"])

print(f"合并后（+ CRSP permno）: {len(panel):,} 条")
print(f"  permno 匹配率: {panel['permno'].notna().mean():.1%}")

In [ ]:
# ── 5.3 为每个 filing 预计算 CRSP 事件窗口数据可用性 ──────────────────────────
# 这一步不计算 CAR（留给 Week 3），只检查每个事件日附近是否有收益数据

# 构建 CRSP 交易日历
trading_days = crsp_daily[["date"]].drop_duplicates().sort_values("date")
trading_days["td_idx"] = range(len(trading_days))
td_dict = dict(zip(trading_days["date"], trading_days["td_idx"]))

def check_data_availability(event_date, permno, min_pre_days=251, min_post_days=3):
    """
    检查事件日前后是否有足够的 CRSP 数据（用于后续 CAR 计算）
    """
    if pd.isna(permno):
        return False
    td = td_dict.get(pd.Timestamp(event_date))
    if td is None:
        return False
    return (td >= min_pre_days) and (td <= len(trading_days) - min_post_days - 1)

panel["has_crsp"] = panel.apply(
    lambda r: check_data_availability(r["date_filed"], r["permno"]),
    axis=1
)

print(f"有足够 CRSP 数据的样本: {panel['has_crsp'].sum():,} / {len(panel):,}")

---
## Part 6: 数据清洗 & 质量检查

In [ ]:
# ── 6.1 样本筛选 ───────────────────────────────────────────────────────────────

print(f"原始记录: {len(panel):,}")

# 只保留主体 10-K（排除 10-K405, 10-KSB 如有需要）
# panel = panel[panel["form_type"] == "10-K"]

# 条件1：MD&A 文本长度合理（太短可能是提取失败，太长可能是误提取）
len_q01 = panel["mda_len"].quantile(0.01)
len_q99 = panel["mda_len"].quantile(0.99)
panel = panel[(panel["mda_len"] >= len_q01) & (panel["mda_len"] <= len_q99)]
print(f"  → 过滤 MD&A 长度异常后: {len(panel):,}")

# 条件2：有 permno（能匹配 CRSP）
panel = panel[panel["permno"].notna()]
print(f"  → 过滤无 permno 后: {len(panel):,}")

# 条件3：有基本面数据
panel = panel[panel["log_assets"].notna()]
print(f"  → 过滤无 Compustat 数据后: {len(panel):,}")

# 条件4：每家公司每年最多一份 10-K
panel = panel.sort_values(["gvkey", "fyear", "date_filed"])
panel = panel.drop_duplicates(subset=["gvkey", "fyear"], keep="last")
print(f"  → 去重（每公司每年一份）后: {len(panel):,}")

print(f"\n✅ 最终样本: {len(panel):,} 个 firm-year 观测")
print(f"   唯一公司: {panel['gvkey'].nunique():,}")
print(f"   年份分布:")
print(panel.groupby("fyear").size().to_string())

In [ ]:
# ── 6.2 描述性统计 ─────────────────────────────────────────────────────────────

desc_vars = ["log_assets", "log_mktcap", "bm_ratio", "roa", "leverage", "mda_len"]
desc = panel[desc_vars].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("描述性统计（最终样本）：")
print(desc.round(3).to_string())

In [ ]:
# ── 6.3 行业分布可视化 ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 行业分布
industry_cnt = panel.groupby("ff_industry").size().sort_values(ascending=False)
industry_cnt.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("行业分布 (FF-12)", fontsize=13)
axes[0].set_xlabel("Filing 数量")

# 年度分布
panel.groupby("fyear").size().plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("年度 Filing 数量", fontsize=13)
axes[1].set_xlabel("财年")
axes[1].tick_params(axis="x", rotation=45)

# MD&A 文本长度分布
panel["mda_words"] = panel["mda_len"] / 5   # 粗估词数（1词≈5字符）
panel["mda_words"].clip(0, 60000).hist(ax=axes[2], bins=50, color="seagreen", edgecolor="white")
axes[2].set_title("MD&A 估计词数分布", fontsize=13)
axes[2].set_xlabel("词数（估计）")
axes[2].axvline(panel["mda_words"].median(), color="red", linestyle="--", label="中位数")
axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week1_summary_stats.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 图表已保存")

---
## Part 7: 保存最终数据集

In [ ]:
# ── 保存 master panel ───────────────────────────────────────────────────────────

# 整理列顺序
final_cols = [
    # 标识符
    "gvkey", "permno", "cik", "ticker", "company_name",
    # 时间
    "fyear", "date_filed", "accession_no",
    # 文本路径
    "mda_path", "mda_len",
    # 行业
    "sic", "ff_industry",
    # 控制变量
    "log_assets", "log_mktcap", "bm_ratio", "roa", "leverage",
]
final_cols = [c for c in final_cols if c in panel.columns]
master = panel[final_cols].copy()

# 保存为 parquet（后续 Week 2/3 加载用）
master.to_parquet(OUTPUT_DIR / "master_panel.parquet", index=False)

# 同时保存一份 CSV 方便查阅（不含长文本路径的精简版）
slim_cols = [c for c in final_cols if c != "mda_path"]
master[slim_cols].to_csv(OUTPUT_DIR / "master_panel_slim.csv", index=False)

print("✅ 最终数据集保存完成：")
print(f"   {OUTPUT_DIR / 'master_panel.parquet'}")
print(f"   {OUTPUT_DIR / 'master_panel_slim.csv'}")
print(f"\n样本规模: {len(master):,} 个 firm-year 观测")
print(f"唯一公司: {master['gvkey'].nunique():,}")
print(f"年份范围: {master['fyear'].min()} – {master['fyear'].max()}")
print(f"\n列说明：")
print(master.dtypes.to_string())

In [ ]:
# ── 断开 WRDS 连接 ──────────────────────────────────────────────────────────────
db.close()
print("✅ WRDS 连接已关闭")

print("""
========================================
  Week 1 完成！数据摘要：
========================================
输出文件：
  data/raw/sp1500_universe.csv         ← S&P 1500 公司名单
  data/raw/compustat_fundamentals.parquet ← 公司基本面
  data/raw/crsp_daily.parquet          ← 日频收益（~300MB）
  data/raw/ccm_link.parquet            ← gvkey-permno 对照
  data/raw/mda_metadata.parquet        ← MD&A 下载状态
  data/filings/                        ← MD&A 文本文件
  data/processed/master_panel.parquet  ← 最终 master panel

下一步（Week 2）：
  1. L&M Dictionary → 计算 negative/uncertain ratio
  2. FinBERT / LLaMA → 批量 inference MD&A 文本
""")

---
## 附录: Midway3 / 本地环境配置备忘

### WRDS 远程连接（Midway3 上）
```bash
# 在 Midway3 上安装 wrds
pip install wrds --user

# 第一次运行时会在 ~/.pgpass 里写入密码
python -c "import wrds; db = wrds.Connection('your_username')"
```

### EDGAR 大规模下载建议
```bash
# SLURM job array：把 CIK 列表分成 N 份并行下载
# 每个 job 处理 ~200 家公司
sbatch --array=0-99 download_mda.sh
```

### 数据大小估计
| 数据 | 估计大小 |
|------|----------|
| CRSP 日频（2009-2021, 1500股）| ~300-500 MB |
| MD&A 文本（每份约 20KB）| ~1-2 GB total |
| Compustat | ~50 MB |
| Master Panel | ~50 MB |